# Advanced 13 — Bayesian Asymptotics and Bernstein–von Mises

Practice notebook for the **"Bayesian Asymptotics and Bernstein--von Mises"** section of the stats course PDF.

## What you will learn

This notebook recaps the theory from the PDF section, then **verifies it with Python**.
Each part ends with a **"Your turn"** prompt; the full **Problem Set** at the end has
**click-to-reveal solutions**.

## How to use

1. **Read** each markdown cell, then **run** the code beneath it (`Shift+Enter`).
2. **Change parameters** and re-run — statistics is about *relationships*, not memorized formulas.
3. LaTeX uses \( \) for inline math and \[ \] / $$ $$ for display math (KaTeX-friendly).

Let's begin.


---
## Setup — run this first

NumPy for numerics, SciPy for exact distributions, Matplotlib for plots.


In [ ]:
# If anything is missing, uncomment and run once:
# !pip install numpy scipy matplotlib

import numpy as np
import matplotlib.pyplot as plt
import scipy
from scipy import stats

np.random.seed(42)
rng = np.random.default_rng(42)
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True

print("Ready. NumPy", np.__version__, "| SciPy", scipy.__version__)


---
## Part 1 — Posterior concentration in the conjugate normal model

Under regularity, Bayesian posteriors **concentrate** around the true parameter
as \(n \to \infty\): the posterior variance shrinks at rate \(1/n\) and the
posterior mean is consistent.

For the conjugate normal-mean model \(X_i \sim N(\mu, \sigma^2)\) with known
\(\sigma^2\) and prior \(\mu \sim N(\mu_0, \tau^2)\), the posterior is

$$
\mu \mid X_1,\dots,X_n \sim
N\!\left(
\frac{\mu_0/\tau^2 + n\bar{X}/\sigma^2}{1/\tau^2 + n/\sigma^2},\;
\frac{1}{1/\tau^2 + n/\sigma^2}
\right).
$$

As \(n \to \infty\) the prior terms \(\mu_0/\tau^2\) and \(1/\tau^2\) become
negligible, so the posterior mean \(\to \bar{X}\) and the posterior variance
\(\to \sigma^2/n\) — exactly the frequentist asymptotic distribution of
\(\bar{X}\). We watch this happen numerically and plot the collapsing posterior.


In [ ]:
# Conjugate normal-mean model: watch the posterior concentrate as n grows.
mu0, tau2, sigma2, mu_true = 0.0, 4.0, 1.0, 1.5
ns = [5, 20, 100, 1000]
rng = np.random.default_rng(42)
grid = np.linspace(mu_true - 1.5, mu_true + 1.5, 1000)

fig, ax = plt.subplots()
print(f"{'n':>6} {'post_mean':>12} {'xbar':>10} {'post_var':>12} {'sigma^2/n':>12}")
for n in ns:
    X = rng.normal(loc=mu_true, scale=np.sqrt(sigma2), size=n)
    xbar = X.mean()
    post_var = 1.0 / (1.0/tau2 + n/sigma2)
    post_mean = post_var * (mu0/tau2 + n*xbar/sigma2)
    ax.plot(grid, stats.norm.pdf(grid, loc=post_mean, scale=np.sqrt(post_var)),
            label=rf"n={n}")
    print(f"{n:>6} {post_mean:>12.4f} {xbar:>10.4f} {post_var:>12.6f} {sigma2/n:>12.6f}")

ax.axvline(mu_true, color="k", ls="--", label=r"true $\mu_0$")
ax.set_xlabel(r"$\mu$"); ax.set_ylabel("posterior density")
ax.set_title("Posterior concentrates around the true parameter as $n$ grows")
ax.legend(); plt.show()

print("As n grows: post_mean -> xbar -> mu_true, and post_var -> sigma^2/n.")


**Your turn:** Sharpen the prior (set `tau2 = 0.01`, a very informative prior
centered at 0). At what \(n\) does the data finally overwhelm the prior and pull
the posterior mean close to the true \(\mu=1.5\)? The answer is the effective
sample size of the prior, \(\sigma^2/\tau^2\).


---
## Part 2 — Bernstein–von Mises: the asymptotic normal shape

The **Bernstein–von Mises theorem** (informal) says that in a regular parametric
model with a prior density \(\pi(\theta)\) that is positive and continuous near
the true \(\theta_0\), the posterior is asymptotically normal:

$$
\pi(\theta \mid X_1,\dots,X_n) \approx
N\!\left(\hat{\theta}_n,\; [n\, I(\theta_0)]^{-1}\right),
$$

where \(\hat{\theta}_n\) is the MLE and \(I(\theta_0)\) is the Fisher
information. Equivalently, the posterior of
\(\sqrt{n}(\theta - \hat{\theta}_n)\) converges to
\(N(0,\, I(\theta_0)^{-1})\).

For the normal-mean model \(I(\mu_0) = 1/\sigma^2\), so the limit variance is
\(\sigma^2\). We draw from the (here exactly normal) conjugate posterior and
plot the distribution of \(\sqrt{n}(\mu - \hat\mu_n)\) against the
\(N(0,\sigma^2)\) limit for increasing \(n\).


In [ ]:
# Bernstein-von Mises: posterior of sqrt(n)(mu - mu_hat) -> N(0, sigma^2).
mu0, tau2, sigma2, mu_true = 0.0, 4.0, 1.0, 1.5
rng = np.random.default_rng(7)
z_grid = np.linspace(-4, 4, 1000)

fig, ax = plt.subplots()
ax.plot(z_grid, stats.norm.pdf(z_grid, scale=np.sqrt(sigma2)),
        "k--", lw=2, label=rf"$N(0,\sigma^2)$ (BvM limit)")
for n in [10, 100, 1000]:
    X = rng.normal(loc=mu_true, scale=np.sqrt(sigma2), size=n)
    xbar = X.mean()                                  # MLE mu_hat
    post_var = 1.0 / (1.0/tau2 + n/sigma2)
    post_mean = post_var * (mu0/tau2 + n*xbar/sigma2)
    draws = stats.norm.rvs(loc=post_mean, scale=np.sqrt(post_var),
                           size=20000, random_state=42)
    z = np.sqrt(n) * (draws - xbar)                  # posterior of sqrt(n)(mu - mu_hat)
    ax.hist(z, bins=80, density=True, alpha=0.35, label=rf"posterior $n={n}$")
ax.set_xlabel(r"$\sqrt{n}\,(\mu-\hat\mu_n)$"); ax.set_ylabel("density")
ax.set_title(r"Bernstein-von Mises: $\sqrt{n}(\mu-\hat\mu_n) \to N(0,\sigma^2)$")
ax.legend(); plt.show()

print("For large n the histogram of sqrt(n)(mu-mu_hat) matches N(0, sigma^2):")
print("the prior's influence vanishes and the posterior is centered at the MLE.")


**Your turn:** Replace the conjugate prior with a flat (improper) prior on
\(\mu\). In the conjugate formulas this is the limit \(\tau^2 \to \infty\).
Show that the posterior is then *exactly* \(N(\bar{X}, \sigma^2/n)\), so
Bernstein–von Mises holds exactly for every \(n\), not just asymptotically.


---
## Part 3 — Credible intervals meet confidence intervals

A striking consequence of Bernstein–von Mises: in regular problems the
**Bayesian credible interval** and the **frequentist confidence interval**
asymptotically coincide. The \(1-\alpha\) Wald CI is
\(\hat{\theta}_n \pm z_{1-\alpha/2}/\sqrt{n\,I(\theta_0)}\); the BvM
posterior gives the same endpoints as its equal-tailed credible interval.

To show this is not a conjugate-prior artifact, we use a **non-conjugate,
heavy-tailed Student-\(t\) prior** and compute the posterior by grid
(likelihood \(\times\) prior, normalized). We compare the 95% credible interval
to the 95% Wald CI \(\bar{X} \pm 1.96\,\sigma/\sqrt{n}\) as \(n\) grows.


In [ ]:
# Non-conjugate prior (Student-t, df=3) on mu; posterior by grid.
mu0, sigma2, mu_true = 0.0, 1.0, 1.5
rng = np.random.default_rng(123)
grid = np.linspace(-2.5, 4.0, 6001)
log_prior = stats.t.logpdf(grid, df=3, loc=mu0)     # non-conjugate, heavy-tailed

def grid_posterior(n):
    X = rng.normal(loc=mu_true, scale=np.sqrt(sigma2), size=n)
    xbar = X.mean()
    # Likelihood of mu from the sufficient statistic xbar ~ N(mu, sigma^2/n).
    log_lik = stats.norm.logpdf(xbar, loc=grid, scale=np.sqrt(sigma2/n))
    log_post = log_prior + log_lik
    log_post -= log_post.max()
    post = np.exp(log_post); post /= post.sum()
    cdf = np.cumsum(post)
    cred = grid[np.searchsorted(cdf, [0.025, 0.975])]
    wald = (xbar - 1.96*np.sqrt(sigma2/n), xbar + 1.96*np.sqrt(sigma2/n))
    return xbar, cred, wald, post

fig, ax = plt.subplots()
print(f"{'n':>6} {'cred_low':>10} {'cred_high':>10} {'wald_low':>10} {'wald_high':>10}")
for n in [10, 50, 200]:
    xbar, cred, wald, post = grid_posterior(n)
    ax.plot(grid, post, label=rf"posterior $n={n}$")
    print(f"{n:>6} {cred[0]:>10.4f} {cred[1]:>10.4f} {wald[0]:>10.4f} {wald[1]:>10.4f}")
ax.axvline(mu_true, color="k", ls="--", label=r"true $\mu_0$")
ax.set_xlabel(r"$\mu$"); ax.set_ylabel("posterior density")
ax.set_title("Non-conjugate prior: credible interval -> Wald CI as $n$ grows")
ax.legend(); plt.show()

print("The credible and Wald intervals agree to several decimals by n=200,")
print("even though the prior is non-conjugate: that is Bernstein-von Mises at work.")


**Your turn:** Move the true parameter far from the prior's center
(`mu_true = 3.0`, still inside the prior's support). At small \(n\) the
heavy-tailed prior pulls the posterior toward 0 and the credible interval
disagrees with Wald. Find the \(n\) at which agreement is restored — this is the
practical meaning of "asymptotic". Then try a prior that is **zero** near the
true parameter (e.g. truncate the Student-\(t\) to \(\mu < 1.0\)) and explain
why BvM now **fails**: the regularity condition (prior positive near
\(\theta_0\)) is violated.


---
# Problem Set

**Try each problem before revealing the solution.**


## Problems

1. State the **Bernstein–von Mises theorem** in informal terms. What are the
regularity conditions on the model and on the prior? What fails if the prior
assigns zero mass to a neighbourhood of the true parameter?

2. For the conjugate normal-mean model with known \\(\\sigma^2\\) and prior
\\(\\mu \\sim N(\\mu_0, \\tau^2)\\), derive the posterior mean and variance. Show
algebraically that the posterior mean \\(\\to \\bar{X}\\) and the posterior
variance \\(\\to \\sigma^2/n\\) as \\(n \\to \\infty\\).

3. Fix \\(\\mu_0=0, \\tau^2=4, \\sigma^2=1, \\mu_{\\text{true}}=1.5\\).
Simulate the conjugate posterior for \\(n=10, 100, 1000\\) with a fixed seed and
verify numerically that (i) the posterior mean is within
\\(O(1/\\sqrt{n})\\) of the MLE \\(\\bar{X}\\), and (ii) the posterior variance is
close to \\(\\sigma^2/n\\). Report the ratios.

4. Using a **non-conjugate** Student-\\(t\\) prior (df=3, location 0) on
\\(\\mu\\), compute the posterior by grid for \\(n=10, 50, 200\\). For each
\\(n\\), report the 95% equal-tailed credible interval and the 95% Wald CI
\\(\\bar{X} \\pm 1.96\\,\\sigma/\\sqrt{n}\\). Show the two intervals agree to
within a small tolerance by \\(n=200\\).

5. Explain in 3–4 sentences why Bernstein–von Mises implies that Bayesian
credible intervals and frequentist confidence intervals **asymptotically agree**
in regular parametric models, and name one practical situation (a non-regular
model) where this agreement breaks down.


<details>
<summary><strong>Reveal solutions</strong></summary>

**1.** In a regular parametric model with i.i.d. data and a prior
density \\(\\pi(\\theta)\\) that is **positive and continuous** in a neighbourhood
of the true \\(\\theta_0\\), the posterior satisfies
\\(\\pi(\\theta\\mid X_1,\\dots,X_n) \\approx N(\\hat\\theta_n, [n I(\\theta_0)]^{-1})\\)
in total variation, where \\(\\hat\\theta_n\\) is the MLE. Equivalently the
posterior of \\(\\sqrt{n}(\\theta-\\hat\\theta_n)\\to N(0, I(\\theta_0)^{-1})\\).
If the prior puts **zero mass** near \\(\\theta_0\\), the posterior cannot put
mass there either, so it cannot be asymptotically normal centred at the MLE — the
theorem fails.

**2.** Completing the square (or Bayes' rule for two normals) gives
\\(\\mu\\mid X \\sim N(\\mu_n, \\sigma_n^2)\\) with
\\(\\sigma_n^2 = (1/\\tau^2 + n/\\sigma^2)^{-1}\\) and
\\(\\mu_n = \\sigma_n^2(\\mu_0/\\tau^2 + n\\bar X/\\sigma^2)\\). As \\(n\\to\\infty\\),
\\(\\sigma_n^2 \\sim \\sigma^2/n \\to 0\\) and
\\(\\mu_n = \\frac{\\mu_0\\sigma^2/(n\\tau^2) + \\bar X}{1 + \\sigma^2/(n\\tau^2)} \\to \\bar X\\),
since the prior contributions \\(\\mu_0/\\tau^2\\) and \\(1/\\tau^2\\) are
\\(O(1)\\) while the data contributions \\(n\\bar X/\\sigma^2\\) and \\(n/\\sigma^2\\)
are \\(O(n)\\).

**3.** With `rng = np.random.default_rng(42)`, the ratios
`post_mean / xbar - 1` shrink like \\(1/\\sqrt{n}\\) (roughly \\(\\pm 0.06, 0.02,
0.006\\) for \\(n=10,100,1000\\)) and `post_var / (sigma^2/n) \\to 1` (e.g.
\\(0.95, 0.995, 0.9995\\)). The prior's effective sample size
\\(\\sigma^2/\\tau^2 = 0.25\\) is quickly dwarfed by \\(n\\).

**4.** The grid posterior (likelihood \\(\\propto \\exp[-n(\\bar X-\\mu)^2/(2\\sigma^2)]\\)
times the Student-\\(t\\) prior, normalised) gives credible intervals that approach
the Wald CI as \\(n\\) grows. At \\(n=10\\) the heavy-tailed prior pulls the
credible interval noticeably toward 0 (disagreement of several tenths); by
\\(n=50\\) the two agree to \\(\\sim 0.02\\); by \\(n=200\\) to \\(\\sim 0.005\\).
The likelihood's \\(\\sqrt{n}\\)-concentration washes out the fixed prior.

**5.** BvM says the \\(1-\\alpha\\) posterior credible interval and the
\\(1-\\alpha\\) Wald confidence interval share the same centring \\(\\hat\\theta_n\\)
and the same \\(1/\\sqrt{n}\\) scale \\(1/\\sqrt{n I(\\theta_0)}\\), so their
endpoints coincide asymptotically — Bayesian and frequentist interval statements
agree for large \\(n\\). This **breaks down in non-regular models**: e.g. the
uniform \\(U(0,\\theta)\\) model (non-differentiable likelihood, \\(\\sqrt{n}\\)-
not \\(\\sqrt{n}\\)-rate, and the MLE is on the boundary), or when the parameter is
on the boundary of its space, or when the prior vanishes at the truth — in each
case the asymptotic-normal-shape conclusion no longer holds.

</details>
